# fewshot-inspector — full MVTec AD benchmark

Reproduces every number in the README: AnomalyDINO-S (672 px) on 15 categories × k ∈ {1, 2, 4, 8, 16} × 3 seeds, the latency pass, the synthetic-defect cutoff calibration, and the report.

**Colab:** *Runtime → Change runtime type → GPU (T4)*, then *Runtime → Run all*. **Kaggle:** enable a GPU accelerator and internet (optionally attach the `ipythonx/mvtec-ad` dataset), then *Run all*. No edits needed.

The MVTec AD dataset (≈5.3 GB) is downloaded from its public mirror and SHA-256 verified. It is © MVTec Software GmbH, **CC BY-NC-SA 4.0** (non-commercial); cite Bergmann et al., CVPR 2019 / IJCV 2021.

In [ ]:
import os, subprocess, sys

def sh(*cmd):
    print('$', ' '.join(cmd), flush=True)
    subprocess.run(list(cmd), check=True)

if not os.path.exists('fewshot-inspector/pyproject.toml'):
    sh('git', 'clone', '--depth', '1', 'https://github.com/SrihaasaK/fewshot-inspector.git')
os.chdir('fewshot-inspector')
sh(sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[bench]')

import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU — this will be slow')

In [ ]:
from pathlib import Path

kaggle = [p for p in (Path('/kaggle/input/mvtec-ad'), Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection'))
          if (p / 'bottle').is_dir()]
if kaggle and not Path('data/mvtec_ad').exists():
    Path('data').mkdir(exist_ok=True)
    os.symlink(kaggle[0], 'data/mvtec_ad')
if Path('data/mvtec_ad/bottle').is_dir() or Path('data/mvtec_ad/mvtec_anomaly_detection/bottle').is_dir():
    sh(sys.executable, 'scripts/download_mvtec.py', '--dest', 'data/mvtec_ad', '--verify-only')
else:
    sh(sys.executable, 'scripts/download_mvtec.py', '--dest', 'data/mvtec_ad')

In [ ]:
# Phase 2: image + pixel AUROC for 15 categories × 5 shots × 3 seeds, then the batch-1 latency pass
sh(sys.executable, 'scripts/benchmark.py', '--categories', 'all', '--shots', '1', '2', '4', '8', '16',
   '--seeds', '0', '1', '2')

In [ ]:
# Phase 3: reference-only cutoffs (synthetic defects) evaluated on the real test sets
sh(sys.executable, 'scripts/calibrate_eval.py', '--categories', 'all', '--shots', '1', '2', '4', '8', '16',
   '--seeds', '0', '1', '2')

In [ ]:
# Failure analysis: per-defect-type AUROC + heatmap grids for the 3 weakest categories (k = 4, seed 0)
sh(sys.executable, 'scripts/failure_cases.py', '--k', '4', '--seed', '0')

In [ ]:
sh(sys.executable, 'scripts/report.py')
from IPython.display import Markdown, display
display(Markdown(Path('results/RESULTS.md').read_text().replace('](', '](results/')))

In [ ]:
import shutil

archive = shutil.make_archive('fewshot_results', 'zip', '.', 'results')
print('results archive:', os.path.abspath(archive))
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Not on Colab: download the archive from the file browser (Kaggle: Output tab).')